<a href="https://colab.research.google.com/github/rahafabumwise/IEEE-AI-Modeling-Hackathon-2.0-Stage-1-Challenge/blob/main/%23%20Experiment%2001%20%E2%80%94%20Hidden%20operating%20regimes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final exact model — Kaggle score 0.20500

This notebook contains only the pipeline used by the successful submission:

- thermal-residual features
- missing-value indicators and median imputation
- stress engineered features
- CatBoost with 1840 iterations
- LightGBM with 1396 iterations
- 80% CatBoost + 20% LightGBM blend in log space

Run every cell from top to bottom.


In [1]:
!pip install -q catboost lightgbm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


## 1. Load data

Change only `DATA_PATH` when needed.


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/datasets/ieee-ai-modeling-hackathon2-stage-1-challenge"

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Test shape:", test.shape)

assert train.shape[0] == 24000
assert test.shape[0] == 16000
assert "edi" in train.columns
assert "id" in train.columns
assert "id" in test.columns


Train shape: (24000, 44)
Test shape: (16000, 43)


In [5]:
X_raw = train.drop(columns=["id", "edi"]).copy()
X_test_raw = test.drop(columns=["id"]).copy()

y_raw = train["edi"].copy()
y_log = np.log1p(y_raw)

print("X_raw:", X_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_log:", y_log.shape)


X_raw: (24000, 42)
X_test_raw: (16000, 42)
y_log: (24000,)


## 2. Exact preprocessing and feature engineering


In [6]:
MISSING_COLS = [
    "humidity",
    "sensor_17",
    "vibration_rms",
    "coolant_flow",
    "hours_since_service",
    "sensor_05"
]


def preprocess_fold(X_train_fold, X_valid_fold):
    X_train_fold = X_train_fold.copy()
    X_valid_fold = X_valid_fold.copy()

    for col in MISSING_COLS:
        X_train_fold[f"{col}_was_missing"] = (
            X_train_fold[col].isna().astype(int)
        )

        X_valid_fold[f"{col}_was_missing"] = (
            X_valid_fold[col].isna().astype(int)
        )

    X_train_fold["total_missing_count"] = (
        X_train_fold[MISSING_COLS].isna().sum(axis=1)
    )

    X_valid_fold["total_missing_count"] = (
        X_valid_fold[MISSING_COLS].isna().sum(axis=1)
    )

    fold_medians = {}

    for col in MISSING_COLS:
        median_value = X_train_fold[col].median()
        fold_medians[col] = median_value

        X_train_fold[col] = X_train_fold[col].fillna(median_value)
        X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

    return X_train_fold, X_valid_fold, fold_medians


In [7]:
def add_engineered_features_stress(X):
    X = X.copy()

    X["load_duty_combo"] = (
        X["load_factor"] * X["duty_cycle"]
    )

    X["stress_index"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
    )

    X["thermal_excess"] = X["delta_ambient"]

    X["electrical_stress"] = (
        X["harmonic_thd"]
        * X["load_factor"]
    )

    X["mechanical_stress"] = (
        X["vibration_rms"]
        * X["load_factor"]
    )

    X["combined_operating_stress"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
        * (1 + X["harmonic_thd"])
    )

    return X


In [8]:
TEMP_PREDICTOR_FEATURES = [
    "asset_age",
    "load_factor",
    "duty_cycle",
    "coolant_flow",
    "humidity",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "harmonic_thd",
    "phase_imbalance",
    "hours_since_service",
    "chamber_pressure"
]


def add_temperature_residual_features(
    X_train_raw,
    X_valid_raw
):
    X_train_raw = X_train_raw.copy()
    X_valid_raw = X_valid_raw.copy()

    temperature_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ])

    temperature_model.fit(
        X_train_raw[TEMP_PREDICTOR_FEATURES],
        X_train_raw["core_temp"]
    )

    train_expected_temp = temperature_model.predict(
        X_train_raw[TEMP_PREDICTOR_FEATURES]
    )

    valid_expected_temp = temperature_model.predict(
        X_valid_raw[TEMP_PREDICTOR_FEATURES]
    )

    X_train_raw["expected_core_temp"] = train_expected_temp
    X_valid_raw["expected_core_temp"] = valid_expected_temp

    X_train_raw["core_temp_residual"] = (
        X_train_raw["core_temp"]
        - X_train_raw["expected_core_temp"]
    )

    X_valid_raw["core_temp_residual"] = (
        X_valid_raw["core_temp"]
        - X_valid_raw["expected_core_temp"]
    )

    X_train_raw["abs_core_temp_residual"] = (
        X_train_raw["core_temp_residual"].abs()
    )

    X_valid_raw["abs_core_temp_residual"] = (
        X_valid_raw["core_temp_residual"].abs()
    )

    return X_train_raw, X_valid_raw, temperature_model


## 3. Build the exact final feature tables


In [9]:
X_full_train_raw = X_raw.copy()
X_full_test_raw = X_test_raw.copy()

(
    X_full_train_thermal,
    X_full_test_thermal,
    final_temperature_model
) = add_temperature_residual_features(
    X_full_train_raw,
    X_full_test_raw
)

(
    X_full_train,
    X_full_test,
    full_medians
) = preprocess_fold(
    X_full_train_thermal,
    X_full_test_thermal
)

X_full_train = add_engineered_features_stress(X_full_train)
X_full_test = add_engineered_features_stress(X_full_test)

assert list(X_full_train.columns) == list(X_full_test.columns)
assert X_full_train.isna().sum().sum() == 0
assert X_full_test.isna().sum().sum() == 0

print("Final training shape:", X_full_train.shape)
print("Final test shape:", X_full_test.shape)

assert X_full_train.shape == (24000, 58)
assert X_full_test.shape == (16000, 58)


Final training shape: (24000, 58)
Final test shape: (16000, 58)


## 4. Train the exact final CatBoost model

The original successful notebook used **1840 iterations**.


In [10]:
final_thermal_model = CatBoostRegressor(
    iterations=1840,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_thermal_model.fit(
    X_full_train,
    y_log
)

thermal_test_log = final_thermal_model.predict(
    X_full_test
)

print("CatBoost predictions:", len(thermal_test_log))
print("CatBoost missing:", np.isnan(thermal_test_log).sum())
print("CatBoost infinite:", np.isinf(thermal_test_log).sum())


0:	learn: 0.9283741	total: 67ms	remaining: 2m 3s
100:	learn: 0.2744276	total: 1.49s	remaining: 25.6s
200:	learn: 0.2143198	total: 2.72s	remaining: 22.2s
300:	learn: 0.1983686	total: 3.93s	remaining: 20.1s
400:	learn: 0.1901752	total: 5.16s	remaining: 18.5s
500:	learn: 0.1843760	total: 6.37s	remaining: 17s
600:	learn: 0.1801740	total: 7.6s	remaining: 15.7s
700:	learn: 0.1767324	total: 8.81s	remaining: 14.3s
800:	learn: 0.1737308	total: 10s	remaining: 13s
900:	learn: 0.1710598	total: 12s	remaining: 12.5s
1000:	learn: 0.1685288	total: 13.7s	remaining: 11.5s
1100:	learn: 0.1661713	total: 14.9s	remaining: 10s
1200:	learn: 0.1638472	total: 16.1s	remaining: 8.58s
1300:	learn: 0.1616742	total: 17.4s	remaining: 7.2s
1400:	learn: 0.1595701	total: 19.1s	remaining: 5.99s
1500:	learn: 0.1575931	total: 20.6s	remaining: 4.64s
1600:	learn: 0.1555828	total: 21.8s	remaining: 3.25s
1700:	learn: 0.1537211	total: 23s	remaining: 1.88s
1800:	learn: 0.1518733	total: 25.1s	remaining: 543ms
1839:	learn: 0.15112

## 5. Train the exact final LightGBM model

The original successful notebook used **1396 iterations**.


In [11]:
thermal_lgb_final_model = LGBMRegressor(
    objective="regression",
    n_estimators=1396,
    learning_rate=0.02,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

thermal_lgb_final_model.fit(
    X_full_train,
    y_log
)

thermal_lgb_test_log = thermal_lgb_final_model.predict(
    X_full_test
)

print("LightGBM predictions:", len(thermal_lgb_test_log))
print("LightGBM missing:", np.isnan(thermal_lgb_test_log).sum())
print("LightGBM infinite:", np.isinf(thermal_lgb_test_log).sum())


LightGBM predictions: 16000
LightGBM missing: 0
LightGBM infinite: 0


## 6. Exact 80/20 log-space blend


In [12]:
thermal_blend_test_log = (
    0.80 * thermal_test_log
    + 0.20 * thermal_lgb_test_log
)

thermal_blend_test_pred = np.clip(
    np.expm1(thermal_blend_test_log),
    0,
    None
)


## 7. Create the submission


In [13]:
submission_thermal_blend = pd.DataFrame({
    "id": test["id"].values,
    "edi": thermal_blend_test_pred
})

submission_thermal_blend.to_csv(
    "submission_thermal_cat80_lgb20.csv",
    index=False
)

print("Submission shape:", submission_thermal_blend.shape)
print(
    "Missing predictions:",
    submission_thermal_blend["edi"].isna().sum()
)
print(
    "Infinite predictions:",
    np.isinf(submission_thermal_blend["edi"]).sum()
)
print(
    "Duplicate IDs:",
    submission_thermal_blend["id"].duplicated().sum()
)
print(
    "Minimum prediction:",
    submission_thermal_blend["edi"].min()
)
print(
    "Maximum prediction:",
    submission_thermal_blend["edi"].max()
)

display(submission_thermal_blend.head())

assert submission_thermal_blend.shape == (16000, 2)
assert submission_thermal_blend["edi"].isna().sum() == 0
assert np.isfinite(submission_thermal_blend["edi"]).all()
assert submission_thermal_blend["id"].equals(test["id"])


Submission shape: (16000, 2)
Missing predictions: 0
Infinite predictions: 0
Duplicate IDs: 0
Minimum prediction: 3.9652839915435703
Maximum prediction: 737.3295358422752


,id,edi
0,24000,46.171586
1,24001,37.690626
2,24002,51.375100
3,24003,78.267724
4,24004,31.575184


The original successful run produced approximately:

- minimum prediction: `3.9652839915`
- maximum prediction: `737.3295358423`
- first prediction: `46.171586`

Small floating-point differences can occur across library versions.


In [14]:
from google.colab import files
files.download("submission_thermal_cat80_lgb20.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>